# Lab 4 — Chat With YOUR Notes
**Session 4 · Embeddings + RAG · TCE** — the weekend's main build (~60 lines, all yours)

You need: your 2–3 documents (PDF or .txt). Upload via folder icon. **File → Save a copy in Drive** first.

In [ ]:
# Cell 1 — setup
%pip install -q -U google-genai pypdf numpy
from getpass import getpass
from google import genai
from google.genai import types
import numpy as np, time, re

client = genai.Client(api_key=getpass("Gemini API key: "))
MODEL = "gemini-flash-latest"  # the free tier's current Flash (July 2026 → Gemini 3.5 Flash). 503 'high demand'? swap to "gemini-flash-lite-latest".
EMBED_MODEL = "gemini-embedding-2"   # check https://ai.google.dev/gemini-api/docs/embeddings

def ask(contents, temperature=0.0):
    for attempt in range(4):
        try:
            return client.models.generate_content(
                model=MODEL, contents=contents,
                config=types.GenerateContentConfig(temperature=temperature)).text
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print("rate limited..."); time.sleep(20*(attempt+1))
            else: raise
print("ready ✓")

## Part A — Ingest: load → chunk → embed → store

> **Keep documents small** — a few pages or one chapter (a few thousand words). Everything runs in Google's cloud, so your laptop's speed/RAM don't matter; this cap just keeps you comfortably inside the free daily quota.

In [ ]:
# Cell 2 — load your document
from pypdf import PdfReader

FILENAME = "my_notes.pdf"   # ← your file (.pdf or .txt)

if FILENAME.endswith(".pdf"):
    text = "\n".join(page.extract_text() or "" for page in PdfReader(FILENAME).pages)
else:
    text = open(FILENAME, encoding="utf-8").read()

print(len(text), "characters loaded")
print(text[:400])   # sanity: is this YOUR text, readable?

In [ ]:
# Cell 3 — chunk: paragraphs, merged to a target size, with overlap
def chunk_text(text, target=800, overlap=150):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) > target and cur:
            chunks.append(cur.strip())
            cur = cur[-overlap:] + " " + p     # overlap keeps thoughts intact
        else:
            cur += " " + p
    if cur.strip(): chunks.append(cur.strip())
    return chunks

chunks = chunk_text(text)

# Keep it laptop- and free-tier-friendly: a few dozen chunks is plenty to learn RAG.
# (Big textbook? Use one chapter. You can always raise this later.)
MAX_CHUNKS = 60
if len(chunks) > MAX_CHUNKS:
    print(f"note: {len(chunks)} chunks -> capping to first {MAX_CHUNKS} (stays well inside the free tier).")
    chunks = chunks[:MAX_CHUNKS]

print(len(chunks), "chunks")
print("--- sample chunk ---\n", chunks[len(chunks)//2][:300])

In [ ]:
# Cell 4 — embed all chunks (batched), store as one numpy matrix
def embed(texts):
    res = client.models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in res.embeddings])

vecs = []
B = 20
for i in range(0, len(chunks), B):
    vecs.append(embed(chunks[i:i+B]))
    print(f"embedded {min(i+B, len(chunks))}/{len(chunks)}")
    time.sleep(1)   # be polite to the free tier

chunk_vecs = np.vstack(vecs)
chunk_vecs = chunk_vecs / np.linalg.norm(chunk_vecs, axis=1, keepdims=True)  # normalize once
print("vector store:", chunk_vecs.shape, "← this numpy array IS your vector database")

### ✓ Checkpoint 1 — chunk count + vector store shape printed.

---
## Part B — Semantic search (the R in RAG)

In [ ]:
# Cell 5 — search = one matrix multiply
def search(query, k=3):
    qv = embed([query])[0]
    qv = qv / np.linalg.norm(qv)
    scores = chunk_vecs @ qv                  # cosine similarity, all chunks at once
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in top]

# sanity check with 3 queries about YOUR material:
for q in ["<your test query 1>", "<query 2>", "<query 3>"]:
    print("=" * 60, "\nQ:", q)
    for s, c in search(q):
        print(f"  {s:.2f} | {c[:110]}...")

### Do the top chunks LOOK right?
If not: chunks too big/small (tune `target`)? PDF extracted garbage (check Cell 2 output)? Query too vague?

### ✓ Checkpoint 2 — three sane searches.

---
## Part C — The full RAG loop (the A and G)

In [ ]:
# Cell 6 — grounded, cited answers
RAG_TEMPLATE = """Answer the question using ONLY the context below.
Cite which chunk you used, like [1] or [2].
If the answer is not in the context, reply exactly: "I don't know based on the provided documents."

CONTEXT:
{context}

QUESTION: {question}"""

def rag_ask(question, k=3, show_chunks=False):
    hits = search(question, k)
    context = "\n\n".join(f"[{i+1}] {c}" for i, (s, c) in enumerate(hits))
    if show_chunks:
        for i, (s, c) in enumerate(hits): print(f"  [{i+1}] ({s:.2f}) {c[:80]}...")
    return ask(RAG_TEMPLATE.format(context=context, question=question))

print(rag_ask("<a real question about your material>", show_chunks=True))

In [ ]:
# Cell 7 — interrogate your own notes (5 real questions)
for q in [
    "<question 1>", "<question 2>", "<question 3>", "<question 4>", "<question 5>",
]:
    print("=" * 60, "\nQ:", q, "\n")
    print(rag_ask(q), "\n")

## Part D — Break it honestly

1. Ask something **definitely NOT in your documents** → does it say "I don't know"? (If it invents instead, strengthen the ONLY/escape-hatch lines — this is real prompt hardening.)
2. Ask something whose answer is **split across two places** → does it get half the truth?

### ✓ Checkpoint 3 — one honest failure + what you changed to fix (or why it's hard).

---
## Stretch goals

In [ ]:
# Stretch 1 — RAG eval (your S2 harness, now grading your app)
rag_tests = [
    {"q": "<question>", "expected": "<key fact from YOUR docs>"},
    # 4 more...
]
def norm(s): return re.sub(r"[^a-z0-9 ]", "", s.lower())
hits = 0
for t in rag_tests:
    ans = rag_ask(t["q"])
    ok = norm(t["expected"]) in norm(ans); hits += ok
    print("✓" if ok else "✗", t["q"])
print(f"RAG score: {hits}/{len(rag_tests)}")

In [ ]:
# Stretch 2 — does k matter?
q = "<a question needing broad context>"
for k in [1, 3, 5]:
    print(f"===== k={k} =====")
    print(rag_ask(q, k=k)[:300], "\n")
# Small k: may miss context. Big k: noise + tokens. Where's YOUR sweet spot?

### S3 · Rerank — retrieve wide, then narrow

The single biggest RAG upgrade after chunking, in about ten lines and no new library. Run it on a question your plain search gets *slightly* wrong — the interesting result is a chunk climbing from position 9 into the top 4.

### S4 · Two scores, never one



In [ ]:
# Stretch 3 — rerank: retrieve wide, then narrow (the biggest upgrade after chunking)
# Your search() is a BI-ENCODER: question and chunks were embedded separately, which is what
# makes it fast. It is good at RECALL (the right chunk is usually in the top 20) and mediocre
# at RANKING (it may sit at position 9 while you only take 3). A reranker reads the question
# and each chunk TOGETHER and re-orders them. Here it is, in one model call.
import json as _json

def rerank(query, candidates, keep=4):
    """candidates: list of (score, chunk). Returns the best `keep`, model-ordered."""
    listing = "\n\n".join(f"[{i}] {c[:400]}" for i, (s, c) in enumerate(candidates))
    prompt = f"""Rank these passages by how well they help answer the question.
Return ONLY the ids of the {keep} most useful, best first.

QUESTION: {query}

PASSAGES:
{listing}"""
    raw = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",     # ← schema, not begging (Session 3)
            response_schema={"type":"object",
                             "properties":{"ids":{"type":"array","items":{"type":"integer"}}},
                             "required":["ids"]})).text
    ids = _json.loads(raw)["ids"][:keep]
    return [candidates[i] for i in ids if 0 <= i < len(candidates)]

q = "<a question your plain search gets slightly wrong>"
wide = search(q, k=20)                  # cheap + approximate
best = rerank(q, wide, keep=4)          # expensive + accurate, on 20 items only

print("BEFORE (top 4 by embedding):")
for s, c in wide[:4]:  print(f"  {s:.2f} | {c[:90]}...")
print("\nAFTER (reranked):")
for s, c in best:      print(f"  {s:.2f} | {c[:90]}...")


In [ ]:
# Stretch 4 — measure retrieval and generation SEPARATELY (two scores, never one)
# Label which chunk SHOULD win for each question, then you can tell WHERE you are broken:
#   low recall@k        -> ingest/chunking is broken (no prompt can save you)
#   good recall, low MRR -> ranking is broken       -> add the reranker above
#   good retrieval, bad answers -> grounding/prompt is broken
probe = [
    # (question, a distinctive phrase that appears in the chunk that SHOULD be retrieved)
    ("<question 1>", "<exact phrase from the right chunk>"),
    # 4 more — this takes 5 minutes and pays for itself immediately
]
K = 5
recall = rr = 0
for q, needle in probe:
    hits = [c for s, c in search(q, k=K)]
    rank = next((i + 1 for i, c in enumerate(hits) if needle.lower() in c.lower()), None)
    recall += rank is not None
    rr     += 1 / rank if rank else 0
    print(f"{'✓' if rank else '✗'} rank={rank}  {q[:52]}")
n = len(probe)
print(f"\nrecall@{K} = {recall}/{n} = {recall/n:.0%}   <- your CEILING on answer accuracy")


## Capstone foundation — saved?

**File → Save.** This notebook returns in Sessions 5 and 6: it gains tools after lunch and gets attacked (then hardened) in the finale. Short break — then AI that *does things*.